# Lead-Lag Analysis

This notebook analyzes the lead-lag relationship between two markets by computing
rolling cross-correlation of price movements at various lags.

The process is as follows:
1. **Load Execution Prices**: Fetch execution prices for both the lead and lag markets.
2. **Resample & Align**: Resample to fixed intervals and align both series to the overlapping time range.
3. **Rolling Lead-Lag Analysis**: Compute cross-correlation in sliding windows to detect which market leads.
4. **Visualize Results**: Plot best lag over time, average correlation by lag, and a correlation heatmap.

## Setup

Import libraries and define the core parameters for the analysis.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import timedelta
from qate.core.model import Market, ExchangeName, Symbol
from qate.env import sys_env
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange
from lead_lag.lead_lag import (
    align_series,
    compute_cross_correlation,
    resample_and_forward_fill,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
%matplotlib inline

PARQUET_STORE_ROOT_DIR = sys_env.get_parquet_root_dir()

LEAD_MARKET = Market(ExchangeName.BINANCE, Symbol.XRP_USDT)
LAG_MARKET = Market(ExchangeName.GMO, Symbol.XRP_JPY)
EXECUTION_SIZE = "150"

DATA_START_DATE = "20250804"
DATA_END_DATE = "20250808"
DATA_DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)

# Lead-lag parameters
WINDOW_SIZE_SEC = 60
WINDOW_STEP_SEC = 30
MAX_LAG = 10
INTERVAL_MS = 500
RESAMPLE_INTERVAL = "100ms"

print(f"Setup complete. Analysis range: {DATA_DT_RANGE.start_str} to {DATA_DT_RANGE.end_str}")
print(f"Lead market: {LEAD_MARKET.id}")
print(f"Lag market:  {LAG_MARKET.id}")
print(f"Window: {WINDOW_SIZE_SEC}s, Step: {WINDOW_STEP_SEC}s, Max lag: {MAX_LAG}, Interval: {INTERVAL_MS}ms")

## 1. Load Execution Prices

Load execution price data for both markets. We add padding around the date range
to ensure we have enough data for the first rolling windows.

In [ ]:
read_range = DtRange(
    DATA_DT_RANGE.start_dt - timedelta(hours=1),
    DATA_DT_RANGE.end_dt + timedelta(hours=1),
)

parquet_store = ParquetStore(PARQUET_STORE_ROOT_DIR)

print(f"Loading execution prices for {LEAD_MARKET.id} (size: {EXECUTION_SIZE})...")
df_lead = parquet_store.create_execution_price(LEAD_MARKET, execution_size_str=EXECUTION_SIZE).read(read_range)
print(f"  Loaded {len(df_lead)} rows")
display(df_lead.head())

print(f"\nLoading execution prices for {LAG_MARKET.id} (size: {EXECUTION_SIZE})...")
df_lag = parquet_store.create_execution_price(LAG_MARKET, execution_size_str=EXECUTION_SIZE).read(read_range)
print(f"  Loaded {len(df_lag)} rows")
display(df_lag.head())

## 2. Resample & Align

Resample both series to fixed intervals and align them to the overlapping time range.
This ensures the cross-correlation is computed on evenly-spaced, comparable data.

In [ ]:
df_lead_resampled = resample_and_forward_fill(df_lead.copy(), RESAMPLE_INTERVAL, "ts")
df_lag_resampled = resample_and_forward_fill(df_lag.copy(), RESAMPLE_INTERVAL, "ts")

print(f"After resampling:")
print(f"  Lead ({LEAD_MARKET.id}): {len(df_lead_resampled)} rows")
print(f"  Lag  ({LAG_MARKET.id}):  {len(df_lag_resampled)} rows")

timestamps, aligned_lead, aligned_lag = align_series(df_lead_resampled, df_lag_resampled)

print(f"\nAligned series length: {len(aligned_lead)}")
print(f"Time range: {timestamps[0]} to {timestamps[-1]}")
print(f"Duration: {timestamps[-1] - timestamps[0]}")

# Plot both mid price series on dual y-axis
fig, ax1 = plt.subplots(figsize=(15, 6))

color_lead = 'tab:blue'
ax1.set_xlabel('Time')
ax1.set_ylabel(f'{LEAD_MARKET.id} mid', color=color_lead)
ax1.plot(timestamps, aligned_lead, color=color_lead, linewidth=0.5, alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color_lead)

ax2 = ax1.twinx()
color_lag = 'tab:red'
ax2.set_ylabel(f'{LAG_MARKET.id} mid', color=color_lag)
ax2.plot(timestamps, aligned_lag, color=color_lag, linewidth=0.5, alpha=0.8)
ax2.tick_params(axis='y', labelcolor=color_lag)

plt.title('Aligned Mid Prices')
fig.tight_layout()
plt.show()

## 3. Rolling Lead-Lag Analysis

Compute cross-correlation in sliding windows across the entire aligned series.
For each window, we find the lag that maximizes the correlation of directional price changes.

In [ ]:
window_size_steps = WINDOW_SIZE_SEC * 1000 // INTERVAL_MS
window_step_steps = WINDOW_STEP_SEC * 1000 // INTERVAL_MS

print(f"Window size: {window_size_steps} steps, Step: {window_step_steps} steps")
print(f"Expected windows: ~{(len(aligned_lead) - window_size_steps) // window_step_steps}")

results = []
start_idx = 0
while start_idx + window_size_steps <= len(aligned_lead):
    end_idx = start_idx + window_size_steps

    window_lead = aligned_lead[start_idx:end_idx]
    window_lag = aligned_lag[start_idx:end_idx]
    window_end_time = timestamps[end_idx - 1]

    best_lag, best_corr, corr_by_lag = compute_cross_correlation(
        window_lead, window_lag, MAX_LAG
    )

    if best_lag is not None:
        row = {
            'window_end_time': window_end_time,
            'best_lag': best_lag * INTERVAL_MS,
            'best_corr': best_corr,
        }
        for lag_idx, corr_val in corr_by_lag.items():
            row[f'lag_{lag_idx * INTERVAL_MS}'] = corr_val
        results.append(row)

    start_idx += window_step_steps

results_df = pd.DataFrame(results)
print(f"\nComputed {len(results_df)} windows")
print(f"\nBest lag summary (ms):")
print(results_df['best_lag'].describe())
print(f"\nBest correlation summary:")
print(results_df['best_corr'].describe())
display(results_df.head(10))

## 4. Visualize Results

In [ ]:
# Plot best_lag and best_corr over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax1.plot(results_df['window_end_time'], results_df['best_lag'], linewidth=0.8, color='tab:blue')
ax1.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax1.set_ylabel('Best Lag (ms)')
ax1.set_title(f'Rolling Lead-Lag: {LEAD_MARKET.id} vs {LAG_MARKET.id}')
ax1.fill_between(results_df['window_end_time'], results_df['best_lag'], 0,
                  where=results_df['best_lag'] > 0, alpha=0.3, color='green', label='Lead leads')
ax1.fill_between(results_df['window_end_time'], results_df['best_lag'], 0,
                  where=results_df['best_lag'] < 0, alpha=0.3, color='red', label='Lag leads')
ax1.legend()

ax2.plot(results_df['window_end_time'], results_df['best_corr'], linewidth=0.8, color='tab:orange')
ax2.set_ylabel('Best Correlation')
ax2.set_xlabel('Time')
ax2.axhline(y=results_df['best_corr'].mean(), color='red', linestyle='--',
            linewidth=0.5, label=f"Mean: {results_df['best_corr'].mean():.3f}")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Average correlation by lag
lag_columns = sorted(
    [c for c in results_df.columns if c.startswith('lag_')],
    key=lambda c: int(c.split('_')[1])
)
lag_ms_values = [int(c.split('_')[1]) for c in lag_columns]
avg_corr = [results_df[c].mean() for c in lag_columns]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if v >= max(avg_corr) * 0.95 else 'steelblue' for v in avg_corr]
ax.bar(range(len(lag_ms_values)), avg_corr, color=colors)
ax.set_xticks(range(len(lag_ms_values)))
ax.set_xticklabels([str(v) for v in lag_ms_values], rotation=45)
ax.set_xlabel('Lag (ms)')
ax.set_ylabel('Average Correlation')
ax.set_title('Average Cross-Correlation by Lag')
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)

best_idx = avg_corr.index(max(avg_corr))
ax.annotate(f'Best: {lag_ms_values[best_idx]}ms\n(corr={max(avg_corr):.4f})',
            xy=(best_idx, max(avg_corr)), xytext=(best_idx + 2, max(avg_corr)),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of correlation over time and lag
heatmap_data = results_df[lag_columns].copy()
heatmap_data.index = results_df['window_end_time']
heatmap_data.columns = lag_ms_values

# Downsample for readability if too many windows
max_display = 200
if len(heatmap_data) > max_display:
    step = len(heatmap_data) // max_display
    heatmap_data = heatmap_data.iloc[::step]

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    heatmap_data.T,
    cmap='RdBu_r',
    center=0,
    ax=ax,
    xticklabels=max(1, len(heatmap_data) // 20),
    yticklabels=True,
    cbar_kws={'label': 'Correlation'},
)
ax.set_xlabel('Time')
ax.set_ylabel('Lag (ms)')
ax.set_title(f'Cross-Correlation Heatmap: {LEAD_MARKET.id} vs {LAG_MARKET.id}')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()